In [17]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import os
import qlbm
from qlbm.initial_conditions import *
from qlbm.visualization import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# ── Lattice ──────────────────────────────────────────────────────────
EXPERIMENT_NAME = "BoundaryConditions2D_TopWall"
NUM_ITERATIONS  = 50

# D2Q5
links          = [[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1]]
weights        = [1/3, 1/6, 1/6, 1/6, 1/6]
speed_of_sound = 1 / np.sqrt(3)

# ── Grid & initial conditions ─────────────────────────────────────────
GRID_SIZE = (8, 16)
Nx, Ny = GRID_SIZE

initial_density = np.full(GRID_SIZE, 0.05)
initial_density[Nx // 2, :] = 0.9

# Zero out wall sites: top wall + top 2 cells of side walls
initial_density[:, -1] = 0.0          # top wall
initial_density[0, -3:] = 0.0         # left side stub (2 cells)
initial_density[-1, -3:] = 0.0        # right side stub (2 cells)

# ── Boundary conditions ────────────────────────────────────────────────
# Minimal BC: top wall + 2 cells of each side wall near the top corner.
nv = len(links)
bc = np.zeros((Nx, Ny, nv, nv))
for x in range(Nx):
    for y in range(Ny):
        bc[x, y] = np.eye(nv)

link_dirs = {tuple(l): i for i, l in enumerate(links)}
STAT = link_dirs[(0, 0)]    # 0
RIGHT = link_dirs[(1, 0)]   # 1
LEFT = link_dirs[(-1, 0)]   # 2
UP = link_dirs[(0, 1)]      # 3
DOWN = link_dirs[(0, -1)]   # 4

REFLECT_FRAC = 0.2

def apply_wall(bc, x, y, perp_in, perp_out, blocked):
    """Redirect perp_in: reflect_frac back, rest split among safe tangentials."""
    safe = [t for t in range(nv) if t not in (perp_in, perp_out, STAT) and t not in blocked]
    bc[x, y, perp_in, perp_in] = 0.0
    bc[x, y, perp_out, perp_in] = REFLECT_FRAC
    if safe:
        frac = (1.0 - REFLECT_FRAC) / len(safe)
        for t in safe:
            bc[x, y, t, perp_in] = frac
    else:
        bc[x, y, perp_out, perp_in] = 1.0

SIDE_WALL_DEPTH = 2  # how many cells of side wall below the top corner

# Top wall: BC nodes at y=Ny-2, for all interior x
for x in range(1, Nx - 1):
    apply_wall(bc, x, Ny - 2, UP, DOWN, blocked=set())

# Left side wall stub: BC nodes at x=1, for top few rows
for y in range(Ny - 2, Ny - 2 - SIDE_WALL_DEPTH, -1):
    blocked = {UP} if y == Ny - 2 else set()  # corner: UP already redirected
    apply_wall(bc, 1, y, LEFT, RIGHT, blocked=blocked)

# Right side wall stub: BC nodes at x=Nx-2, for top few rows
for y in range(Ny - 2, Ny - 2 - SIDE_WALL_DEPTH, -1):
    blocked = {UP} if y == Ny - 2 else set()
    apply_wall(bc, Nx - 2, y, RIGHT, LEFT, blocked=blocked)

boundary_conditions = bc

# ── Velocity field — uniform upward push ───────────────────────────────
velocity = np.zeros((2, *GRID_SIZE))
velocity[1, :, :] = 0.3

config = [
    (NUM_ITERATIONS, velocity, links, weights, speed_of_sound, boundary_conditions),
]

# Quick sanity-check plot
fig, ax = plt.subplots(figsize=(5, 8))
im = ax.imshow(initial_density.T, cmap='viridis', origin='lower')
plt.colorbar(im, ax=ax, label='Density')
ax.set_title('Initial density — top wall + side stubs')
ax.set_xlabel('X'); ax.set_ylabel('Y')
plt.tight_layout(); plt.show()

In [ ]:
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
dirname    = f"experiments/{EXPERIMENT_NAME}_{timestamp}"
os.makedirs(dirname, exist_ok=True)

classical_csv = f"{dirname}/classical.csv"
quantum_csv   = f"{dirname}/quantum.csv"

qlbm.simulate_flow_classical(initial_density, config, classical_csv)
qlbm.simulate_flow(initial_density, config, quantum_csv)

Classical simulation: iterations 0-50/50
Classical Iteration 1/50
Classical Iteration 2/50
Classical Iteration 3/50
Classical Iteration 4/50
Classical Iteration 5/50
Classical Iteration 6/50
Classical Iteration 7/50
Classical Iteration 8/50
Classical Iteration 9/50
Classical Iteration 10/50
Classical Iteration 11/50
Classical Iteration 12/50
Classical Iteration 13/50
Classical Iteration 14/50
Classical Iteration 15/50
Classical Iteration 16/50
Classical Iteration 17/50
Classical Iteration 18/50
Classical Iteration 19/50
Classical Iteration 20/50
Classical Iteration 21/50
Classical Iteration 22/50
Classical Iteration 23/50
Classical Iteration 24/50
Classical Iteration 25/50
Classical Iteration 26/50
Classical Iteration 27/50
Classical Iteration 28/50
Classical Iteration 29/50
Classical Iteration 30/50
Classical Iteration 31/50
Classical Iteration 32/50
Classical Iteration 33/50
Classical Iteration 34/50
Classical Iteration 35/50
Classical Iteration 36/50
Classical Iteration 37/50
Classi

In [26]:
# Side-by-side snapshots (classical vs quantum)
snapshot_iters = [0, NUM_ITERATIONS // 4, NUM_ITERATIONS // 2, NUM_ITERATIONS - 1]

df_c = pd.read_csv(classical_csv, header=None)
df_q = pd.read_csv(quantum_csv, header=None)

vmin = min(df_c.values.min(), df_q.values.min())
vmax = max(df_c.values.max(), df_q.values.max())

fig, axes = plt.subplots(len(snapshot_iters), 2, figsize=(10, 4 * len(snapshot_iters)))
fig.suptitle('Classical (left) vs Quantum (right)', fontsize=16, y=1.01)

for row, t in enumerate(snapshot_iters):
    for col, (df, label) in enumerate([(df_c, 'Classical'), (df_q, 'Quantum')]):
        ax = axes[row, col]
        frame = df.iloc[t].values.reshape(GRID_SIZE, order='F')
        im = ax.imshow(frame.T, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
        ax.set_title(f'{label}  t={t}')
        ax.set_xlabel('X'); ax.set_ylabel('Y')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Density')
plt.tight_layout(rect=[0, 0, 0.88, 1.0])
plt.savefig(f"{dirname}/snapshots.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Snapshots saved to {dirname}/snapshots.png")

FileNotFoundError: [Errno 2] No such file or directory: 'experiments/BoundaryConditions2D_Line_SplitWalls_20260407_131627/quantum.csv'

In [ ]:
# RMSE between classical and quantum over time
qlbm.save_rmse_comparison(classical_csv, quantum_csv, GRID_SIZE, f"{dirname}/rmse.png")

In [ ]:
# Animated classical result
animate_density_evolution(GRID_SIZE, classical_csv, interval=100, boundary_nodes=boundary_mask(boundary_conditions))

In [ ]:
# Animated quantum result
animate_density_evolution(GRID_SIZE, quantum_csv, interval=100, boundary_nodes=boundary_mask(boundary_conditions))

In [ ]:
# Mass conservation check
classical_mass = df_c.sum(axis=1)
quantum_mass = df_q.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(classical_mass, label='Classical')
ax.plot(quantum_mass, label='Quantum', linestyle='--')
ax.set_xlabel('Iteration')
ax.set_ylabel('Total mass')
ax.set_title('Mass conservation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{dirname}/mass_conservation.png", dpi=300, bbox_inches='tight')
plt.show()